In [18]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [19]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

device = torch.device('cuda')
sys.path.append('/media/carsen/ssd1/github/oneshot')

In [20]:
from utils import data
mouse_id = 4
depth_separable = True
pool = True
clamp = True
data_path = '../data'

# change path to your own dm11 path
img_root = '../data'
weight_path = '../weights/'

In [21]:
# load images
img = data.load_images(img_root, file=os.path.join(img_root, data.db[mouse_id]["stim"]), crop=True)
nimg, Ly, Lx = img.shape
print('img: ', img.shape, img.min(), img.max(), img.dtype)

raw image shape:  (15032, 66, 264)
cropped image shape:  (15032, 66, 130)
image mean:  127.91226
image std:  61.150326
img:  (15032, 66, 130) -2.0917675 2.078284 float32


In [22]:
# load neurons
fname = '%s_nat15k_%s.npz'%(data.db[mouse_id]['mname'], data.db[mouse_id]['datexp'])
spks, istim_train, istim_test, xpos, ypos, spks_rep_all, _ = data.load_neurons(file_path = os.path.join(data_path, fname), mouse_id = mouse_id)
n_stim, n_neurons = spks.shape
print('spks: ', spks.shape, spks.min(), spks.max())
print('spks_rep_all: ', len(spks_rep_all), spks_rep_all[0].shape)
print('istim_train: ', istim_train.shape, istim_train.min(), istim_train.max())
print('istim_test: ', istim_test.shape, istim_test.min(), istim_test.max())

# selec the subset of training set for hyperparam search
n_train = 2000
val_idxes = np.random.choice(np.where(istim_train)[0], n_train, replace=False)
istim_train = istim_train[val_idxes]
spks = spks[val_idxes]


loading activities from ../data/TX115_nat15k_2024_01_08.npz
spks:  (5102, 36036) 0.0 53.338394
spks_rep_all:  500 (2, 36036)
istim_train:  (5102,) 532 5599
istim_test:  (500,) 32 531


In [23]:
# split train and validation set
itrain, ival = data.split_train_val(istim_train, train_frac=0.9)
print('itrain: ', itrain.shape, itrain.min(), itrain.max())
print('ival: ', ival.shape, ival.min(), ival.max())


splitting training and validation set...
there is currently no randomness in this function now, please make sure the istim_train is in random order!
itrain:  (1800,)
ival:  (200,)
itrain:  (1800,) 1 1999
ival:  (200,) 0 1990


In [24]:
spks, spks_rep_all = data.normalize_spks(spks, spks_rep_all, itrain)


normalizing neural data...


In [25]:
from utils import metrics
test_fev = metrics.fev_nan(spks_rep_all)
print('FEV (test): ', np.mean(test_fev))

valid_idxes = np.where(test_fev > 0.15)[0]
print('valid_idxes: ', len(valid_idxes))

FEV (test):  0.13790827029603334
valid_idxes:  13209


In [26]:
# ineur = np.arange(0, n_neurons) #np.arange(0, n_neurons, 5)
# ineur = np.where(test_fev > 0.)[0] # use only neurons with FEV > 0.15
# subsample some neurons
ineur = np.arange(0, n_neurons, 10) # use every 5th neuron
spks_train = torch.from_numpy(spks[itrain][:,ineur])
spks_val = torch.from_numpy(spks[ival][:,ineur]) 
spks_rep_all = [spks_rep_all[i][:,ineur] for i in range(len(spks_rep_all))]
print('spks_train: ', spks_train.shape, spks_train.min(), spks_train.max())
print('spks_val: ', spks_val.shape, spks_val.min(), spks_val.max())

img_train = torch.from_numpy(img[istim_train][itrain]).to(device).unsqueeze(1) # change :130 to 25:100 
img_val = torch.from_numpy(img[istim_train][ival]).to(device).unsqueeze(1)
img_test = torch.from_numpy(img[istim_test]).to(device).unsqueeze(1)

print('img_train: ', img_train.shape, img_train.min(), img_train.max())
print('img_val: ', img_val.shape, img_val.min(), img_val.max())
print('img_test: ', img_test.shape, img_test.min(), img_test.max())

input_Ly, input_Lx = img_train.shape[-2:]

spks_train:  torch.Size([1800, 3604]) tensor(0.) tensor(33.5999)
spks_val:  torch.Size([200, 3604]) tensor(0.) tensor(38.6161)
img_train:  torch.Size([1800, 1, 66, 130]) tensor(-2.0918, device='cuda:0') tensor(2.0783, device='cuda:0')
img_val:  torch.Size([200, 1, 66, 130]) tensor(-2.0918, device='cuda:0') tensor(2.0783, device='cuda:0')
img_test:  torch.Size([500, 1, 66, 130]) tensor(-2.0918, device='cuda:0') tensor(2.0783, device='cuda:0')


In [27]:
train_real_responses = torch.ones_like(spks_train)
val_real_responses = torch.ones_like(spks_val)
# set nans to zero
train_real_responses[torch.isnan(spks_train)] = 0
val_real_responses[torch.isnan(spks_val)] = 0
spks_train[torch.isnan(spks_train)] = 0
spks_val[torch.isnan(spks_val)] = 0

In [ ]:
# build model
from utils import model_builder, model_trainer
nlayers = 2
nconv1 = 16
nconv2 = 64

# Bayesian optimization

In [30]:
import optuna

def objective(trial):
    lr = trial.suggest_loguniform("lr", 1e-4, 1e-2)
    wd_core = trial.suggest_loguniform("weight_decay_core", 1e-3, 1.0)
    l2_readout = trial.suggest_loguniform("l2_readout", 1e-3, 1.0)

    model, in_channels = model_builder.build_model(NN=len(ineur), n_layers=nlayers, n_conv=nconv1, n_conv_mid=nconv2, pool=pool, depth_separable=depth_separable, input_Ly=input_Ly, input_Lx=input_Lx)
    model_name = model_builder.create_model_name(data.db[mouse_id]["mname"], data.db[mouse_id]["datexp"], n_layers=nlayers, in_channels=in_channels, clamp=clamp, ineuron=len(ineur))
    # weight_path = './checkpoints/'
    model_path = os.path.join(weight_path, 'fullmodel', data.db[mouse_id]["mname"], model_name)
    print('modelpath: ', model_path)
    model = model.to(device)

    best_state = model_trainer.monkey_train(
        model,
        spks_train, train_real_responses,
        spks_val, val_real_responses,
        img_train, img_val,
        hs_readout=0,
        l2_readout=l2_readout,
        weight_decay_core=wd_core,
        device=device,
        learning_rate=lr
    )

    model.load_state_dict(best_state)
    model.eval()
    _, varexp = model_trainer.monkey_val_epoch(model, img_val, spks_val, val_real_responses, batch_size=100, device=device)
    return -varexp.mean().item()  # minimize negative variance explained

study = optuna.create_study()
study.optimize(objective, n_trials=20)

[I 2026-04-16 16:25:24,377] A new study created in memory with name: no-name-f3e0d7c9-a81b-43bd-98b9-1570649b35bf
/tmp/ipykernel_1464036/1301059053.py:4: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform("lr", 1e-4, 1e-2)
/tmp/ipykernel_1464036/1301059053.py:5: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  wd_core = trial.suggest_loguniform("weight_decay_core", 1e-3, 1.0)
/tmp/ipykernel_1464036/1301059053.py:6: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  l2_readout = trial.suggest_log

input shape of readout:  (64, 33, 65)
model name:  TX115_2024_01_08_2layer_16_64_clamp_sensorium_depthsep_pool_nneurons_3604.pt
modelpath:  ../weights/fullmodel/TX115/TX115_2024_01_08_2layer_16_64_clamp_sensorium_depthsep_pool_nneurons_3604.pt
0.0026154988654727524
epoch 0, train_loss = 0.0092, val_loss = 0.0083, varexp_val = -0.0319, time 0.74s
epoch 1, train_loss = 0.0079, val_loss = 0.0078, varexp_val = -0.0116, time 1.47s
epoch 2, train_loss = 0.0078, val_loss = 0.0078, varexp_val = -0.0083, time 2.21s
epoch 3, train_loss = 0.0077, val_loss = 0.0078, varexp_val = -0.0086, time 2.93s
epoch 4, train_loss = 0.0077, val_loss = 0.0077, varexp_val = -0.0022, time 3.64s
epoch 5, train_loss = 0.0076, val_loss = 0.0077, varexp_val = 0.0019, time 4.37s
epoch 6, train_loss = 0.0076, val_loss = 0.0077, varexp_val = -0.0038, time 5.09s
epoch 7, train_loss = 0.0076, val_loss = 0.0077, varexp_val = 0.0032, time 5.82s
epoch 8, train_loss = 0.0075, val_loss = 0.0077, varexp_val = 0.0027, time 6.56s

[I 2026-04-16 16:25:53,382] Trial 0 finished with value: -0.0104488180950284 and parameters: {'lr': 0.0026154988654727524, 'weight_decay_core': 0.05209189256210357, 'l2_readout': 0.002951681076342367}. Best is trial 0 with value: -0.0104488180950284.


epoch 4, train_loss = 0.0072, val_loss = 0.0076, varexp_val = 0.0093, time 3.80s
Early stopping at epoch 4 due to no improvement in validation varexp.
input shape of readout:  (64, 33, 65)
model name:  TX115_2024_01_08_2layer_16_64_clamp_sensorium_depthsep_pool_nneurons_3604.pt
modelpath:  ../weights/fullmodel/TX115/TX115_2024_01_08_2layer_16_64_clamp_sensorium_depthsep_pool_nneurons_3604.pt
0.002344303332342144
epoch 0, train_loss = 0.0092, val_loss = 0.0082, varexp_val = -0.0323, time 0.75s
epoch 1, train_loss = 0.0079, val_loss = 0.0079, varexp_val = -0.0181, time 1.49s
epoch 2, train_loss = 0.0078, val_loss = 0.0078, varexp_val = -0.0087, time 2.23s
epoch 3, train_loss = 0.0077, val_loss = 0.0078, varexp_val = -0.0058, time 2.97s
epoch 4, train_loss = 0.0077, val_loss = 0.0077, varexp_val = -0.0004, time 3.71s
epoch 5, train_loss = 0.0076, val_loss = 0.0077, varexp_val = 0.0035, time 4.45s
epoch 6, train_loss = 0.0076, val_loss = 0.0077, varexp_val = -0.0025, time 5.18s
epoch 7, tr

[I 2026-04-16 16:26:22,548] Trial 1 finished with value: -0.010098010301589966 and parameters: {'lr': 0.002344303332342144, 'weight_decay_core': 0.044209332594695096, 'l2_readout': 0.016073479597203014}. Best is trial 0 with value: -0.0104488180950284.


...

epoch 4, train_loss = 0.0073, val_loss = 0.0077, varexp_val = 0.0089, time 3.44s
Early stopping at epoch 4 due to no improvement in validation varexp.
input shape of readout:  (64, 33, 65)
model name:  TX115_2024_01_08_2layer_16_64_clamp_sensorium_depthsep_pool_nneurons_3604.pt
modelpath:  ../weights/fullmodel/TX115/TX115_2024_01_08_2layer_16_64_clamp_sensorium_depthsep_pool_nneurons_3604.pt
0.0016513042936917065
epoch 0, train_loss = 0.0095, val_loss = 0.0083, varexp_val = -0.0944, time 0.69s
epoch 1, train_loss = 0.0080, val_loss = 0.0079, varexp_val = -0.0084, time 1.38s
epoch 2, train_loss = 0.0078, val_loss = 0.0078, varexp_val = -0.0087, time 2.07s
epoch 3, train_loss = 0.0077, val_loss = 0.0078, varexp_val = -0.0069, time 2.75s
epoch 4, train_loss = 0.0077, val_loss = 0.0078, varexp_val = 0.0006, time 3.45s
epoch 5, train_loss = 0.0076, val_loss = 0.0077, varexp_val = 0.0029, time 4.14s
epoch 6, train_loss = 0.0076, val_loss = 0.0078, varexp_val = -0.0103, time 4.82s
epoch 7, tr

[I 2026-04-16 16:34:57,810] Trial 18 finished with value: -0.00978898536413908 and parameters: {'lr': 0.0016513042936917065, 'weight_decay_core': 0.2735125111039444, 'l2_readout': 0.0010061121104573574}. Best is trial 5 with value: -0.010541269555687904.


epoch 4, train_loss = 0.0073, val_loss = 0.0077, varexp_val = 0.0082, time 3.43s
Early stopping at epoch 4 due to no improvement in validation varexp.
input shape of readout:  (64, 33, 65)
model name:  TX115_2024_01_08_2layer_16_64_clamp_sensorium_depthsep_pool_nneurons_3604.pt
modelpath:  ../weights/fullmodel/TX115/TX115_2024_01_08_2layer_16_64_clamp_sensorium_depthsep_pool_nneurons_3604.pt
0.0005944304338768625
epoch 0, train_loss = 0.0099, val_loss = 0.0097, varexp_val = -0.4256, time 0.69s
epoch 1, train_loss = 0.0093, val_loss = 0.0085, varexp_val = -0.1472, time 1.37s
epoch 2, train_loss = 0.0080, val_loss = 0.0080, varexp_val = -0.0137, time 2.06s
epoch 3, train_loss = 0.0078, val_loss = 0.0078, varexp_val = -0.0083, time 2.75s
epoch 4, train_loss = 0.0078, val_loss = 0.0078, varexp_val = -0.0061, time 3.43s
epoch 5, train_loss = 0.0077, val_loss = 0.0078, varexp_val = -0.0044, time 4.12s
epoch 6, train_loss = 0.0077, val_loss = 0.0078, varexp_val = -0.0041, time 4.81s
epoch 7, 

[I 2026-04-16 16:35:33,069] Trial 19 finished with value: -0.005030161235481501 and parameters: {'lr': 0.0005944304338768625, 'weight_decay_core': 0.08520483235574182, 'l2_readout': 0.09096316912943904}. Best is trial 5 with value: -0.010541269555687904.


epoch 5, train_loss = 0.0074, val_loss = 0.0077, varexp_val = 0.0047, time 4.64s
Early stopping at epoch 5 due to no improvement in validation varexp.


In [31]:
print("Best trial:")
best_trial = study.best_trial

print(f"  Value (objective): {-best_trial.value:.4f}")  # negate if you returned -val_varexp
print("  Params:")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")


# Best trial:
#   Value (objective): 0.0745
#   Params:
#     lr: 0.002810981518314036
#     weight_decay_core: 0.0011958116794013951
#     l2_readout: 0.0010818284653872753

Best trial:
  Value (objective): 0.0105
  Params:
    lr: 0.006448216486357653
    weight_decay_core: 0.13690492467837284
    l2_readout: 0.00963153300581214


In [32]:
import joblib
joblib.dump(study, f"optuna_study_{nlayers}layers_nat15k.pkl")

['optuna_study_2layers_nat15k.pkl']

In [33]:
# import joblib
# study = joblib.load("optuna_study_4layers.pkl")

# print("Best trial:")
# best_trial = study.best_trial

# print(f"  Value (objective): {-best_trial.value:.4f}")  # negate if you returned -val_varexp
# print("  Params:")
# for key, value in best_trial.params.items():
#     print(f"    {key}: {value}")

# Best trial:
#   Value (objective): 0.0755
#   Params:
#     lr: 0.0026146339759175433
#     weight_decay_core: 0.06758502313016607
#     l2_readout: 0.12013247147550409

In [ ]:
# lr = [0.006, 0.006, 0.003, 0.003]
# weight_decay_core = [0.1, 0.1, 0.003, 0.06]
# l2_readout = [0.001, 0.01, 0.1, 0.1]

In [35]:
import optuna.visualization as vis

# Plot optimization history
vis.plot_optimization_history(study).show()

# Plot parameter importance (which hyperparams matter most)
vis.plot_param_importances(study).show()

# Parallel coordinate plot (hyperparams vs. objective)
vis.plot_parallel_coordinate(study).show()
